In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
import itertools

from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, utils
from tensorflow.keras.applications.resnet import ResNet152, preprocess_input
from tensorflow.keras.preprocessing import image_dataset_from_directory

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, balanced_accuracy_score, confusion_matrix
)

print("TensorFlow Version:", tf.keras.__version__)

TensorFlow Version: 3.10.0


In [ ]:
data_dir = "/content/drive/MyDrive/Bone Break Classification/Bone Break Classification"

train_data = image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="int",
    validation_split=0.1,
    subset="training",
    shuffle=True,
    seed=40,
    image_size=(256, 256),
    batch_size=64,
    color_mode="rgb"
)

validation_data = image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="int",
    validation_split=0.2,
    subset="validation",
    shuffle=True,
    seed=42,
    image_size=(256, 256),
    batch_size=64,
    color_mode="rgb"
)

class_names = train_data.class_names
NUM_CLASSES = len(class_names)
print("Class names:", class_names)

Found 1129 files belonging to 10 classes.
Using 1017 files for training.
Found 1129 files belonging to 10 classes.
Using 225 files for validation.
Class names: ['Avulsion fracture', 'Comminuted fracture', 'Fracture Dislocation', 'Greenstick fracture', 'Hairline Fracture', 'Impacted fracture', 'Longitudinal fracture', 'Oblique fracture', 'Pathological fracture', 'Spiral Fracture']


In [ ]:
def preprocess_for_model(img):
    img = tf.cast(img, tf.float32)
    return preprocess_input(img)

def preprocess_for_display(img):
    img = tf.cast(img, tf.float32) / 255.0
    return img

train_dataset = train_data.map(lambda x, y: (preprocess_for_model(x), y))
val_dataset   = validation_data.map(lambda x, y: (preprocess_for_model(x), y))
display_dataset = train_data.map(lambda x, y: (preprocess_for_display(x), y))

In [ ]:
def dataset_to_numpy(dataset):
    x_data, y_data = [], []
    for images, labels in dataset:
        x_data.append(images.numpy())
        y_data.append(labels.numpy())
    return np.concatenate(x_data), np.concatenate(y_data)

x_train, y_train = dataset_to_numpy(train_dataset)
x_val, y_val     = dataset_to_numpy(val_dataset)
x_display, y_display = dataset_to_numpy(display_dataset)

In [ ]:
print("Before squeeze:", y_train.shape, y_val.shape)

y_train = np.squeeze(y_train)
y_val   = np.squeeze(y_val)

y_train = utils.to_categorical(y_train, NUM_CLASSES)
y_val   = utils.to_categorical(y_val, NUM_CLASSES)

print("After one-hot:", y_train.shape, y_val.shape)

Before squeeze: (1017,) (225,)
After one-hot: (1017, 10) (225, 10)


In [ ]:
input_layer = layers.Input(shape=(256, 256, 3))

base_model = ResNet152(
    weights="imagenet",
    include_top=False,
    input_tensor=input_layer
)
base_model.trainable = False   # SAME AS EfficientNet

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(520, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output_layer = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model_resnet152 = models.Model(inputs=input_layer, outputs=output_layer)
model_resnet152.summary()

234698864/234698864 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 262, 262,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 128, 128,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 128, 128,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 128, 128,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 130, 130,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 64, 64,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 64, 64,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 64, 64,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 64, 64,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 64, 64,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 64, 64,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 64, 64,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 64, 64,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 64, 64,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 64, 64,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 64, 64,    │      1,024 │ conv2_block1_3_c

 Total params: 59,034,338 (225.20 MB)

 Trainable params: 663,394 (2.53 MB)

 Non-trainable params: 58,370,944 (222.67 MB)

In [ ]:
opt = optimizers.RMSprop(learning_rate=0.0001)

# Removed redundant to_categorical calls as y_train and y_val are already one-hot encoded.

model_resnet152.compile(
    optimizer=opt,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model_resnet152.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=100,
    validation_data=(x_val, y_val),
    shuffle=True
)

Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - accuracy: 0.1121 - loss: 2.6758 - val_accuracy: 0.2267 - val_loss: 2.1838
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 311ms/step - accuracy: 0.1719 - loss: 2.3185 - val_accuracy: 0.2978 - val_loss: 2.1133
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 314ms/step - accuracy: 0.1876 - loss: 2.2693 - val_accuracy: 0.3467 - val_loss: 2.0495
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 321ms/step - accuracy: 0.2276 - loss: 2.2024 - val_accuracy: 0.4089 - val_loss: 1.9912
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 324ms/step - accuracy: 0.2733 - loss: 2.0727 - val_accuracy: 0.4356 - val_loss: 1.9200
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 323ms/step - accuracy: 0.2886 - loss: 2.0627 - val_accuracy: 0.4578 - val_loss: 1.8672
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 327ms/step - accuracy: 0.2912 - loss: 1.9983 - val_accuracy: 0.5156 - val_loss: 1.8010
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 11s 331ms/step - accuracy: 0.3446 - loss: 1.9187 - val

In [ ]:
y_pred_probs = model_resnet152.predict(x_val)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_val, axis=1)

acc   = accuracy_score(y_true_classes, y_pred_classes)
prec  = precision_score(y_true_classes, y_pred_classes, average='weighted')
rec   = recall_score(y_true_classes, y_pred_classes, average='weighted')
f1    = f1_score(y_true_classes, y_pred_classes, average='weighted')
bal_acc = balanced_accuracy_score(y_true_classes, y_pred_classes)

print(f"\nAccuracy          : {acc:.4f}")
print(f"Precision         : {prec:.4f}")
print(f"Recall            : {rec:.4f}")
print(f"F1 Score          : {f1:.4f}")
print(f"Balanced Accuracy : {bal_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))

8/8 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step

Accuracy          : 0.9244
Precision         : 0.9276
Recall            : 0.9244
F1 Score          : 0.9239
Balanced Accuracy : 0.9199

Classification Report:
                       precision    recall  f1-score   support

    Avulsion fracture       0.87      1.00      0.93        20
  Comminuted fracture       0.88      1.00      0.94        29
 Fracture Dislocation       0.91      0.94      0.92        31
  Greenstick fracture       0.95      1.00      0.98        20
    Hairline Fracture       0.93      0.93      0.93        27
    Impacted fracture       1.00      0.83      0.91        18
Longitudinal fracture       0.94      0.89      0.91        18
     Oblique fracture       0.94      0.88      0.91        17
Pathological fracture       0.96      0.87      0.91        30
      Spiral Fracture       0.93      0.87      0.90        15

             accuracy                           0.92       225
            macro avg       0.93      0.92   